# 中证800 V83：V46 标签、排序目标与样本时效性综合实验

本 notebook 在完全固定 `full_v46` 特征、LightGBM 复杂度、训练折叠和 `top8_board_cap` 组合规则的前提下，只比较五种训练机制：

- `A0_raw_regression`：V46 原始 `alpha_1m` 回归基线
- `A1_robust_z_regression`：月度截面 robust z-score 标签
- `A2_rank_regression`：月度截面 percentile-rank 标签
- `A3_lambdarank_5grade`：月度五档 relevance + LambdaRank
- `A4_recency_weighted`：原始标签 + 固定24个月半衰期样本权重

评估覆盖 ROC-AUC、PR-AUC、Precision、Recall、MAP、NDCG、RankIC、分桶单调性、Top8 实现收益、fold 稳定性、block bootstrap 和健康指标有效性。所有训练按模型逐一完成并立即释放，避免 JoinQuant OOM。


## 0. 导入、进度条与绘图基础


In [ ]:
import os
import gc
import math
import warnings
import builtins as _bi
from pathlib import Path

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 260)
pd.set_option("display.width", 260)
pd.set_option("display.max_rows", 160)

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None


def progress_iter(iterable, total=None, desc="progress", leave=True):
    if tqdm is not None:
        return tqdm(iterable, total=total, desc=desc, leave=leave)
    def _gen():
        every = _bi.max(1, int((total or 100) / 20))
        for i, item in enumerate(iterable, 1):
            if i == 1 or i % every == 0 or (total is not None and i == total):
                print("%s %s%s" % (desc, i, "/%s" % total if total else ""))
            yield item
    return _gen()


def display_df(df, n=30):
    try:
        display(df.head(n))
    except Exception:
        print(df.head(n).to_string(index=False))


def save_show(fig, filename):
    fig.tight_layout()
    fig.savefig(FIG_DIR / filename, dpi=140, bbox_inches="tight")
    plt.show()
    plt.close(fig)


## 1. 实验配置


In [ ]:
PROJECT_DIR = Path.cwd()
OUT_DIR = PROJECT_DIR / "csi800_ml_v83_objective_label_diagnostics_outputs"
FIG_DIR = OUT_DIR / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

DATA_CANDIDATES = [
    Path("train_csi800_factor_v40_data_enhancement_20190101_20260531.csv"),
    Path("train_csi800_factor_v40_data_enhancement.csv"),
    Path("data/train_csi800_factor_v40_data_enhancement_20190101_20260531.csv"),
    Path("data/train_csi800_factor_v40_data_enhancement.csv"),
    PROJECT_DIR / "train_csi800_factor_v40_data_enhancement_20190101_20260531.csv",
    PROJECT_DIR / "train_csi800_factor_v40_data_enhancement.csv",
]
DATA_PATH_OVERRIDE = None

TARGET_COL = "alpha_1m"
STOCK_COL = "stock"
DATE_COL = "rebalance_date"
INDUSTRY_COL = "industry_bucket"
LABEL_BOUNDARY_MODE = "legacy_rebalance"

FIXED_ITER = 120
SEED = 42
CORR_THRESHOLD = 0.70
RECENCY_HALF_LIFE_MONTHS = 24.0
TOP_N_CANDIDATES = 30
STOCK_NUM = 8
BOARD_CAPS = {"chinext": 3, "star": 2}
BUCKET_N = 10
BOOTSTRAP_N = 1000
BOOTSTRAP_BLOCK_MONTHS = 6

RUN_VARIANTS = [
    "A0_raw_regression",
    "A1_robust_z_regression",
    "A2_rank_regression",
    "A3_lambdarank_5grade",
    "A4_recency_weighted",
]

FOLD_PLAN = [
    {"fold_id": "cutoff202112", "train_start": "2019-01-01", "train_end": "2021-12-31", "test_start": "2022-01-01", "test_end": "2022-12-31"},
    {"fold_id": "cutoff202212", "train_start": "2019-01-01", "train_end": "2022-12-31", "test_start": "2023-01-01", "test_end": "2023-12-31"},
    {"fold_id": "cutoff202312", "train_start": "2019-01-01", "train_end": "2023-12-31", "test_start": "2024-01-01", "test_end": "2024-12-31"},
    {"fold_id": "cutoff202412", "train_start": "2019-01-01", "train_end": "2024-12-31", "test_start": "2025-01-01", "test_end": "2025-12-31"},
    {"fold_id": "cutoff202512", "train_start": "2019-01-01", "train_end": "2025-12-31", "test_start": "2026-01-01", "test_end": "2026-12-31"},
]

SMOKE_TEST = False
if SMOKE_TEST:
    RUN_VARIANTS = RUN_VARIANTS[:2]
    FOLD_PLAN = FOLD_PLAN[:1]
    BOOTSTRAP_N = 100

COLORS = {
    "A0_raw_regression": "#2f5597",
    "A1_robust_z_regression": "#00a087",
    "A2_rank_regression": "#e64b35",
    "A3_lambdarank_5grade": "#7e6148",
    "A4_recency_weighted": "#4dbbd5",
}

print("OUT_DIR:", OUT_DIR)
print("RUN_VARIANTS:", RUN_VARIANTS)
print("FOLDS:", [x["fold_id"] for x in FOLD_PLAN])


## 2. V46 full 特征和固定模型参数


In [ ]:
def unique_keep_order(cols):
    seen = set()
    out = []
    for col in cols:
        if col not in seen:
            out.append(col)
            seen.add(col)
    return out


BASE_FACTOR_COLS = [
    "cash_flow_to_price_ratio", "book_to_price_ratio", "earnings_yield", "sales_to_price_ratio",
    "cash_earnings_to_price_ratio", "earnings_to_price_ratio", "roe_ttm", "roa_ttm",
    "gross_profit_ttm", "operating_profit_to_total_profit", "net_operate_cash_flow_to_total_liability",
    "net_operating_cash_flow_coverage", "adjusted_profit_to_total_profit", "ACCA", "growth",
    "net_working_capital", "operating_profit_per_share", "net_operate_cash_flow_per_share",
    "total_operating_revenue_per_share", "super_quick_ratio", "MLEV", "debt_to_equity_ratio",
    "debt_to_tangible_equity_ratio", "momentum", "Rank1M", "sharpe_ratio_60", "Variance20",
    "liquidity", "beta", "ATR6", "MFI14", "DAVOL10", "VOL10", "VMACD", "VOSC",
    "Skewness20", "Kurtosis20",
]
HYBRID_LIGHT_EXTRA_COLS = [
    "liq_money_ratio_20_60", "liq_paused_count_20", "px_close_to_ma60", "px_drawdown_60",
    "ts_cash_flow_to_price_ratio_rank_mean_3m", "ts_Rank1M_rank_chg_1m",
]
FULL_FEATURE_COLS = unique_keep_order(BASE_FACTOR_COLS + HYBRID_LIGHT_EXTRA_COLS)

BASE_PARAMS_FF10 = {
    "objective": "regression",
    "metric": "l2",
    "boosting_type": "gbdt",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "min_data_in_leaf": 200,
    "feature_fraction": 1.0,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "lambda_l1": 0.1,
    "lambda_l2": 0.3,
    "verbose": -1,
}

VARIANT_MANIFEST = [
    {"variant": "A0_raw_regression", "target_mode": "raw", "objective_mode": "regression", "sample_weight_mode": "equal"},
    {"variant": "A1_robust_z_regression", "target_mode": "robust_z", "objective_mode": "regression", "sample_weight_mode": "equal"},
    {"variant": "A2_rank_regression", "target_mode": "rank", "objective_mode": "regression", "sample_weight_mode": "equal"},
    {"variant": "A3_lambdarank_5grade", "target_mode": "grade5", "objective_mode": "lambdarank", "sample_weight_mode": "equal"},
    {"variant": "A4_recency_weighted", "target_mode": "raw", "objective_mode": "regression", "sample_weight_mode": "recency24"},
]
VARIANT_MANIFEST = [x for x in VARIANT_MANIFEST if x["variant"] in set(RUN_VARIANTS)]
variant_manifest_df = pd.DataFrame(VARIANT_MANIFEST)
variant_manifest_df.to_csv(OUT_DIR / "v83_variant_manifest.csv", index=False)
display_df(variant_manifest_df, 20)


## 3. 数据、标签和训练工具


In [ ]:
def resolve_data_path():
    if DATA_PATH_OVERRIDE:
        p = Path(DATA_PATH_OVERRIDE)
        if p.exists():
            return p
        raise IOError("DATA_PATH_OVERRIDE not found: %s" % p)
    for raw in DATA_CANDIDATES:
        p = Path(raw)
        if p.exists():
            return p
    searched = [str(Path(x).resolve()) for x in DATA_CANDIDATES]
    raise IOError("training data csv not found: %s" % searched)


def safe_to_datetime(df, cols):
    out = df.copy()
    for col in cols:
        if col in out.columns:
            out[col] = pd.to_datetime(out[col], errors="coerce").dt.normalize()
    return out


def safe_rank_ic(a, b):
    s = pd.DataFrame({"a": np.asarray(a, dtype=float), "b": np.asarray(b, dtype=float)})
    s = s.replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) < 3 or s["a"].nunique() < 2 or s["b"].nunique() < 2:
        return np.nan
    return float(s["a"].rank(method="average").corr(s["b"].rank(method="average")))


def safe_pearson_ic(a, b):
    s = pd.DataFrame({"a": np.asarray(a, dtype=float), "b": np.asarray(b, dtype=float)})
    s = s.replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) < 3 or s["a"].nunique() < 2 or s["b"].nunique() < 2:
        return np.nan
    return float(s["a"].corr(s["b"]))


def load_dataset(path):
    df = pd.read_csv(path)
    df = safe_to_datetime(df, [DATE_COL, "feature_date", "next_date"])
    if STOCK_COL not in df.columns:
        for alt in ["code", "security", "order_book_id"]:
            if alt in df.columns:
                df = df.rename(columns={alt: STOCK_COL})
                break
    if TARGET_COL not in df.columns:
        if "raw_return_1m" in df.columns and "benchmark_csi800_1m" in df.columns:
            df[TARGET_COL] = pd.to_numeric(df["raw_return_1m"], errors="coerce") - pd.to_numeric(df["benchmark_csi800_1m"], errors="coerce")
        else:
            raise ValueError("target column not found: " + TARGET_COL)
    if INDUSTRY_COL not in df.columns:
        df[INDUSTRY_COL] = "UNKNOWN"
    if "feature_date" not in df.columns:
        df["feature_date"] = df[DATE_COL]
    if "next_date" not in df.columns:
        df["next_date"] = df[DATE_COL]
    need = [STOCK_COL, DATE_COL, TARGET_COL, INDUSTRY_COL, "feature_date", "next_date"]
    missing = [c for c in need if c not in df.columns]
    if missing:
        raise ValueError("dataset missing columns: " + ",".join(missing))
    missing_features = [c for c in FULL_FEATURE_COLS if c not in df.columns]
    if missing_features:
        raise ValueError("missing full_v46 features: " + ",".join(missing_features))
    df[STOCK_COL] = df[STOCK_COL].astype(str)
    df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
    df = df.dropna(subset=[STOCK_COL, DATE_COL, TARGET_COL]).copy()
    df = df.sort_values([DATE_COL, STOCK_COL]).reset_index(drop=True)
    for col in progress_iter(FULL_FEATURE_COLS + [TARGET_COL], total=len(FULL_FEATURE_COLS) + 1, desc="compact float32"):
        df[col] = pd.to_numeric(df[col], errors="coerce").replace([np.inf, -np.inf], np.nan).astype(np.float32)
    return df


def make_train_df(df, fold):
    start = pd.Timestamp(fold["train_start"])
    end = pd.Timestamp(fold["train_end"])
    mask = (df[DATE_COL] >= start) & (df[DATE_COL] <= end)
    if LABEL_BOUNDARY_MODE == "label_end_safe":
        mask = mask & (df["next_date"] <= end)
    elif LABEL_BOUNDARY_MODE != "legacy_rebalance":
        raise ValueError("unknown LABEL_BOUNDARY_MODE: " + str(LABEL_BOUNDARY_MODE))
    return df.loc[mask].copy()


def make_test_df(df, fold):
    start = pd.Timestamp(fold["test_start"])
    end = pd.Timestamp(fold["test_end"])
    return df.loc[(df[DATE_COL] >= start) & (df[DATE_COL] <= end)].copy()


def build_corr_components(train_df, feature_cols, threshold):
    from collections import defaultdict
    corr = train_df[feature_cols].corr()
    graph = defaultdict(list)
    for i in range(len(feature_cols)):
        for j in range(i + 1, len(feature_cols)):
            v = corr.iloc[i, j]
            if not pd.isnull(v) and abs(v) > threshold:
                graph[feature_cols[i]].append(feature_cols[j])
                graph[feature_cols[j]].append(feature_cols[i])
    for col in feature_cols:
        graph[col]
    visited = set()
    comps = []
    def dfs(x, comp):
        visited.add(x)
        comp.append(x)
        for y in graph[x]:
            if y not in visited:
                dfs(y, comp)
    for col in feature_cols:
        if col not in visited:
            comp = []
            dfs(col, comp)
            comps.append(comp)
    return comps


def select_features_train_only(train_df, candidate_cols):
    cols = unique_keep_order([c for c in candidate_cols if c in train_df.columns])
    missing = train_df[cols].isnull().sum().to_dict()
    keep = []
    remove = []
    for comp in build_corr_components(train_df, cols, CORR_THRESHOLD):
        if len(comp) == 1:
            keep.append(comp[0])
        else:
            ordered = _bi.sorted(comp, key=lambda x: (missing[x], x))
            keep.append(ordered[0])
            remove.extend(ordered[1:])
    if len(keep) == 0:
        raise ValueError("no usable full_v46 feature")
    return keep, remove


def prepare_x(df, feature_cols, fill_values=None):
    X = df.reindex(columns=feature_cols).replace([np.inf, -np.inf], np.nan).copy()
    if fill_values is None:
        fill_values = X.median().replace([np.inf, -np.inf], np.nan).fillna(0)
    X = X.fillna(fill_values).fillna(0)
    return X, fill_values


def transform_monthly_target(train_df, mode):
    out = pd.Series(np.nan, index=train_df.index, dtype=float)
    groups = train_df.groupby(DATE_COL).groups
    for _, idx in groups.items():
        s = pd.to_numeric(train_df.loc[idx, TARGET_COL], errors="coerce")
        valid = s.dropna()
        if len(valid) == 0:
            continue
        if mode == "raw":
            vals = s
        elif mode == "robust_z":
            med = float(valid.median())
            mad = float((valid - med).abs().median()) * 1.4826
            if not np.isfinite(mad) or mad <= 1e-12:
                mad = float(valid.std())
            if not np.isfinite(mad) or mad <= 1e-12:
                mad = 1.0
            vals = ((s - med) / mad).clip(-5.0, 5.0)
        elif mode in ["rank", "grade5"]:
            rank = s.rank(method="average", ascending=True)
            pct = rank / float(len(valid))
            if mode == "rank":
                vals = pct - 0.5
            else:
                vals = np.floor(pct * 5.0).clip(0, 4)
        else:
            raise ValueError("unknown target mode: " + str(mode))
        out.loc[idx] = vals
    return out


def make_recency_weights(train_df, train_end):
    end = pd.Timestamp(train_end)
    age_months = (end - pd.to_datetime(train_df[DATE_COL])).dt.days.astype(float) / 30.4375
    weights = np.power(0.5, age_months / float(RECENCY_HALF_LIFE_MONTHS))
    weights = np.asarray(weights, dtype=float)
    mean_w = float(np.nanmean(weights))
    if not np.isfinite(mean_w) or mean_w <= 0:
        return np.ones(len(train_df), dtype=float)
    return weights / mean_w


def train_variant(train_df, feature_cols, variant, train_end):
    work = train_df.sort_values([DATE_COL, STOCK_COL]).copy()
    y = transform_monthly_target(work, variant["target_mode"])
    valid_mask = y.notnull()
    work = work.loc[valid_mask].copy()
    y = y.loc[valid_mask]
    X, fill_values = prepare_x(work, feature_cols, None)
    params = dict(BASE_PARAMS_FF10)
    params["seed"] = SEED
    weights = None
    group_sizes = None
    if variant["sample_weight_mode"] == "recency24":
        weights = make_recency_weights(work, train_end)
    if variant["objective_mode"] == "lambdarank":
        params["objective"] = "lambdarank"
        params["metric"] = "ndcg"
        params["label_gain"] = [0, 1, 3, 7, 15]
        y = y.astype(int)
        group_sizes = work.groupby(DATE_COL).size().values.astype(int).tolist()
    dataset = lgb.Dataset(X[feature_cols], label=np.asarray(y), weight=weights, group=group_sizes)
    model = lgb.train(params, dataset, num_boost_round=_bi.max(1, int(FIXED_ITER)))
    train_pred = np.asarray(model.predict(X[feature_cols], num_iteration=FIXED_ITER)).reshape(-1)
    train_rank_ic = safe_rank_ic(train_pred, train_df.loc[work.index, TARGET_COL])
    return {"model": model, "fill_values": fill_values, "train_rows": len(work), "train_rank_ic": train_rank_ic}


def score_model(df, trained, feature_cols):
    X, _ = prepare_x(df, feature_cols, trained["fill_values"])
    return np.asarray(trained["model"].predict(X[feature_cols], num_iteration=FIXED_ITER)).reshape(-1)


## 4. 排序、分类、组合和统计指标


In [ ]:
def average_precision_at_k(pred_order, relevant_set, k):
    if len(relevant_set) == 0:
        return np.nan
    hits = 0
    score = 0.0
    top = pred_order[:_bi.min(k, len(pred_order))]
    for i, stock in enumerate(top, 1):
        if stock in relevant_set:
            hits += 1
            score += hits / float(i)
    denom = float(_bi.min(len(relevant_set), k))
    return score / denom if denom > 0 else np.nan


def dcg_at_k(gains, k):
    vals = np.asarray(gains[:_bi.min(k, len(gains))], dtype=float)
    if len(vals) == 0:
        return np.nan
    vals = vals - np.nanmin(vals)
    denom = np.log2(np.arange(2, len(vals) + 2))
    return float(np.nansum(vals / denom))


def ndcg_at_k(pred_order, gain_map, k):
    pred_gains = [gain_map.get(s, np.nan) for s in pred_order]
    ideal_gains = _bi.sorted([v for v in gain_map.values() if not pd.isnull(v)], reverse=True)
    dcg = dcg_at_k(pred_gains, k)
    idcg = dcg_at_k(ideal_gains, k)
    if pd.isnull(dcg) or pd.isnull(idcg) or idcg <= 0:
        return np.nan
    return dcg / idcg


def roc_auc_binary(y_true, scores):
    d = pd.DataFrame({"y": np.asarray(y_true, dtype=float), "s": np.asarray(scores, dtype=float)})
    d = d.replace([np.inf, -np.inf], np.nan).dropna()
    n_pos = int((d["y"] > 0).sum())
    n_neg = int((d["y"] <= 0).sum())
    if n_pos == 0 or n_neg == 0:
        return np.nan
    ranks = d["s"].rank(method="average", ascending=True)
    rank_sum = float(ranks[d["y"] > 0].sum())
    return (rank_sum - n_pos * (n_pos + 1) / 2.0) / float(n_pos * n_neg)


def average_precision_binary(y_true, scores):
    d = pd.DataFrame({"y": np.asarray(y_true, dtype=float), "s": np.asarray(scores, dtype=float)})
    d = d.replace([np.inf, -np.inf], np.nan).dropna().sort_values("s", ascending=False)
    y = (d["y"].values > 0).astype(int)
    n_pos = int(y.sum())
    if n_pos == 0:
        return np.nan
    tp = np.cumsum(y)
    precision = tp / np.arange(1, len(y) + 1, dtype=float)
    return float(precision[y == 1].sum() / n_pos)


def binary_curve_points(y_true, scores):
    d = pd.DataFrame({"y": np.asarray(y_true, dtype=float), "s": np.asarray(scores, dtype=float)})
    d = d.replace([np.inf, -np.inf], np.nan).dropna().sort_values("s", ascending=False)
    y = (d["y"].values > 0).astype(int)
    n_pos = int(y.sum())
    n_neg = int(len(y) - n_pos)
    if n_pos == 0 or n_neg == 0:
        return pd.DataFrame()
    tp = np.cumsum(y).astype(float)
    fp = np.cumsum(1 - y).astype(float)
    recall = tp / float(n_pos)
    precision = tp / np.arange(1, len(y) + 1, dtype=float)
    tpr = recall
    fpr = fp / float(n_neg)
    return pd.DataFrame({"fpr": np.r_[0.0, fpr, 1.0], "tpr": np.r_[0.0, tpr, 1.0], "recall": np.r_[0.0, recall, 1.0], "precision": np.r_[1.0, precision, float(n_pos) / len(y)]})


def board_name(stock):
    code6 = str(stock).split(".")[0]
    if code6.startswith(("300", "301")):
        return "chinext"
    if code6.startswith("688"):
        return "star"
    return "main"


def select_top8_board_cap(month_df):
    ordered = month_df.sort_values("score", ascending=False)[STOCK_COL].astype(str).tolist()
    selected = []
    counts = {"chinext": 0, "star": 0, "main": 0}
    for stock in ordered[:_bi.min(TOP_N_CANDIDATES, len(ordered))]:
        board = board_name(stock)
        cap = BOARD_CAPS.get(board, STOCK_NUM)
        if counts.get(board, 0) >= cap:
            continue
        selected.append(stock)
        counts[board] = counts.get(board, 0) + 1
        if len(selected) >= STOCK_NUM:
            break
    return selected


def calc_nav(ret_series):
    s = pd.Series(ret_series).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return (1.0 + s).cumprod()


def calc_mdd(ret_series):
    nav = calc_nav(ret_series)
    if len(nav) == 0:
        return np.nan
    return float((nav / nav.cummax() - 1.0).min())


def annualized_return(ret_series):
    s = pd.Series(ret_series).replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) == 0:
        return np.nan
    total = float((1.0 + s).prod())
    if total <= 0:
        return np.nan
    return total ** (12.0 / len(s)) - 1.0


def icir(series):
    s = pd.to_numeric(pd.Series(series), errors="coerce").dropna()
    if len(s) < 2 or float(s.std()) <= 1e-12:
        return np.nan
    return float(s.mean() / s.std() * math.sqrt(12.0))


def moving_block_bootstrap_mean(values, n_sim=BOOTSTRAP_N, block=BOOTSTRAP_BLOCK_MONTHS, seed=SEED):
    x = np.asarray(pd.Series(values).dropna(), dtype=float)
    n = len(x)
    if n < 3:
        return np.nan, np.nan, np.nan
    rng = np.random.RandomState(seed)
    block = int(_bi.max(1, _bi.min(block, n)))
    sims = []
    starts = np.arange(n)
    for _ in range(int(n_sim)):
        sampled = []
        while len(sampled) < n:
            st = int(rng.choice(starts))
            sampled.extend([x[(st + j) % n] for j in range(block)])
        sims.append(float(np.mean(sampled[:n])))
    arr = np.asarray(sims, dtype=float)
    return float(np.percentile(arr, 2.5)), float(np.percentile(arr, 97.5)), float((arr > 0).mean())


## 5. 加载与审计数据


In [ ]:
DATA_PATH = resolve_data_path()
df_all = load_dataset(DATA_PATH)
audit_rows = [
    {"check": "rows", "value": int(len(df_all))},
    {"check": "months", "value": int(df_all[DATE_COL].nunique())},
    {"check": "date_min", "value": str(df_all[DATE_COL].min())},
    {"check": "date_max", "value": str(df_all[DATE_COL].max())},
    {"check": "duplicate_stock_date", "value": int(df_all.duplicated([STOCK_COL, DATE_COL]).sum())},
    {"check": "feature_count", "value": int(len(FULL_FEATURE_COLS))},
]
data_audit_df = pd.DataFrame(audit_rows)
data_audit_df.to_csv(OUT_DIR / "v83_data_audit.csv", index=False)
print("DATA_PATH:", DATA_PATH)
print("loaded:", df_all.shape)
display_df(data_audit_df, 20)
display_df(df_all[[TARGET_COL]].describe().T, 5)


## 6. 逐 fold、逐 variant 训练并生成低内存 OOS score panel


In [ ]:
score_parts = []
model_meta_rows = []
total_models = len(FOLD_PLAN) * len(VARIANT_MANIFEST)
model_counter = 0

for fold in progress_iter(FOLD_PLAN, total=len(FOLD_PLAN), desc="walk-forward folds"):
    train_df = make_train_df(df_all, fold)
    test_df = make_test_df(df_all, fold)
    if train_df.empty or test_df.empty:
        print("skip empty fold", fold["fold_id"], train_df.shape, test_df.shape)
        del train_df, test_df
        gc.collect()
        continue
    feature_cols, removed_cols = select_features_train_only(train_df, FULL_FEATURE_COLS)
    for variant in VARIANT_MANIFEST:
        model_counter += 1
        print("train model %s/%s: %s %s" % (model_counter, total_models, fold["fold_id"], variant["variant"]))
        trained = train_variant(train_df, feature_cols, variant, fold["train_end"])
        pred = score_model(test_df, trained, feature_cols)
        keep_cols = [STOCK_COL, DATE_COL, TARGET_COL, INDUSTRY_COL, "next_date"]
        keep_cols = [c for c in keep_cols if c in test_df.columns]
        part = test_df[keep_cols].copy()
        part["score"] = pred.astype(np.float32)
        part["variant"] = variant["variant"]
        part["fold_id"] = fold["fold_id"]
        part["train_end"] = pd.Timestamp(fold["train_end"])
        score_parts.append(part)
        model_meta_rows.append({
            "fold_id": fold["fold_id"], "variant": variant["variant"],
            "train_start": fold["train_start"], "train_end": fold["train_end"],
            "test_start": fold["test_start"], "test_end": fold["test_end"],
            "train_months": int(train_df[DATE_COL].nunique()), "train_rows": int(len(train_df)),
            "test_months": int(test_df[DATE_COL].nunique()), "test_rows": int(len(test_df)),
            "feature_count": int(len(feature_cols)), "removed_feature_count": int(len(removed_cols)),
            "train_rank_ic": trained["train_rank_ic"],
            "target_mode": variant["target_mode"], "objective_mode": variant["objective_mode"],
            "sample_weight_mode": variant["sample_weight_mode"],
            "feature_cols": ",".join(feature_cols), "removed_features": ",".join(removed_cols),
        })
        del trained, pred, part
        gc.collect()
    del train_df, test_df
    gc.collect()

score_panel_df = pd.concat(score_parts, ignore_index=True) if score_parts else pd.DataFrame()
model_meta_df = pd.DataFrame(model_meta_rows)
del score_parts
gc.collect()
model_meta_df.to_csv(OUT_DIR / "v83_model_meta.csv", index=False)
print("score_panel:", score_panel_df.shape)
display_df(model_meta_df, 30)
if len(score_panel_df) == 0:
    raise ValueError("empty OOS score panel")


## 7. 计算月度综合指标、分桶表现与 pooled ROC/PR 曲线


In [ ]:
monthly_rows = []
bucket_rows = []
enriched_parts = []
grouped = score_panel_df.groupby(["variant", "fold_id", DATE_COL])
for (variant, fold_id, dt), gdf in progress_iter(grouped, total=grouped.ngroups, desc="monthly diagnostics"):
    m = gdf.dropna(subset=["score", TARGET_COL]).copy()
    if len(m) < 50:
        continue
    m[STOCK_COL] = m[STOCK_COL].astype(str)
    m = m.sort_values("score", ascending=False).reset_index(drop=True)
    pred_order = m[STOCK_COL].tolist()
    true_order = m.sort_values(TARGET_COL, ascending=False)[STOCK_COL].tolist()
    gain_map = dict(zip(m[STOCK_COL], pd.to_numeric(m[TARGET_COL], errors="coerce")))
    true8 = set(true_order[:8])
    true20 = set(true_order[:20])
    raw8 = pred_order[:8]
    board8 = select_top8_board_cap(m)
    raw8_set = set(raw8)
    board8_set = set(board8)
    y8 = np.asarray([1 if s in true8 else 0 for s in m[STOCK_COL]], dtype=int)
    y20 = np.asarray([1 if s in true20 else 0 for s in m[STOCK_COL]], dtype=int)
    scores = np.asarray(m["score"], dtype=float)
    universe_mean = float(pd.to_numeric(m[TARGET_COL], errors="coerce").mean())
    raw8_alpha = float(np.nanmean([gain_map.get(s, np.nan) for s in raw8]))
    board8_alpha = float(np.nanmean([gain_map.get(s, np.nan) for s in board8]))
    bottom8 = pred_order[-8:]
    bottom8_alpha = float(np.nanmean([gain_map.get(s, np.nan) for s in bottom8]))
    row = {
        "variant": variant, "fold_id": fold_id, DATE_COL: pd.Timestamp(dt), "n_universe": int(len(m)),
        "rank_ic": safe_rank_ic(scores, m[TARGET_COL]),
        "pearson_ic": safe_pearson_ic(scores, m[TARGET_COL]),
        "roc_auc_true_top8": roc_auc_binary(y8, scores),
        "pr_auc_true_top8": average_precision_binary(y8, scores),
        "roc_auc_true_top20": roc_auc_binary(y20, scores),
        "pr_auc_true_top20": average_precision_binary(y20, scores),
        "precision_true_top8_at8_raw": len(raw8_set & true8) / 8.0,
        "recall_true_top8_at8_raw": len(raw8_set & true8) / 8.0,
        "precision_true_top20_at8_raw": len(raw8_set & true20) / 8.0,
        "recall_true_top20_at8_raw": len(raw8_set & true20) / 20.0,
        "precision_true_top20_at8_boardcap": len(board8_set & true20) / float(_bi.max(1, len(board8))),
        "recall_true_top20_at8_boardcap": len(board8_set & true20) / 20.0,
        "map_true_top20_at8": average_precision_at_k(pred_order, true20, 8),
        "ndcg_alpha_at8": ndcg_at_k(pred_order, gain_map, 8),
        "ndcg_alpha_at20": ndcg_at_k(pred_order, gain_map, 20),
        "prevalence_true_top8": 8.0 / len(m),
        "prevalence_true_top20": 20.0 / len(m),
        "precision_lift_top20_at8": (len(raw8_set & true20) / 8.0) / (20.0 / len(m)),
        "universe_alpha": universe_mean,
        "raw_top8_alpha": raw8_alpha,
        "boardcap_top8_alpha": board8_alpha,
        "boardcap_top8_edge": board8_alpha - universe_mean,
        "top_bottom8_spread": raw8_alpha - bottom8_alpha,
        "selected_count": int(len(board8)),
    }
    monthly_rows.append(row)

    n = len(m)
    m["score_rank_pct"] = (np.arange(n, dtype=float) + 1.0) / float(n)
    m["true_top8"] = y8
    m["true_top20"] = y20
    enriched_parts.append(m[["variant", "fold_id", DATE_COL, STOCK_COL, "score_rank_pct", "true_top8", "true_top20"]])

    for bucket in range(1, BUCKET_N + 1):
        lo = int(math.floor((bucket - 1) * n / float(BUCKET_N)))
        hi = int(math.floor(bucket * n / float(BUCKET_N)))
        b = m.iloc[lo:hi]
        bucket_rows.append({
            "variant": variant, "fold_id": fold_id, DATE_COL: pd.Timestamp(dt),
            "score_bucket": bucket, "n": int(len(b)),
            "mean_alpha": float(pd.to_numeric(b[TARGET_COL], errors="coerce").mean()),
        })

monthly_metrics_df = pd.DataFrame(monthly_rows)
bucket_monthly_df = pd.DataFrame(bucket_rows)
curve_panel_df = pd.concat(enriched_parts, ignore_index=True) if enriched_parts else pd.DataFrame()
monthly_metrics_df.to_csv(OUT_DIR / "v83_monthly_metrics.csv", index=False)
bucket_monthly_df.to_csv(OUT_DIR / "v83_bucket_monthly.csv", index=False)
print("monthly metrics:", monthly_metrics_df.shape, "bucket:", bucket_monthly_df.shape)
display_df(monthly_metrics_df, 20)


## 8. 汇总、fold 稳定性、相对 V46 基线和 bootstrap


In [ ]:
def summarize_variant(gdf):
    ret = pd.to_numeric(gdf["boardcap_top8_edge"], errors="coerce").dropna()
    return {
        "months": int(len(gdf)),
        "rank_ic_mean": float(gdf["rank_ic"].mean()),
        "rank_ic_median": float(gdf["rank_ic"].median()),
        "rank_ic_positive_rate": float((gdf["rank_ic"] > 0).mean()),
        "rank_ic_ir_annualized": icir(gdf["rank_ic"]),
        "roc_auc_true_top20_mean": float(gdf["roc_auc_true_top20"].mean()),
        "pr_auc_true_top20_mean": float(gdf["pr_auc_true_top20"].mean()),
        "precision_true_top20_at8_mean": float(gdf["precision_true_top20_at8_raw"].mean()),
        "recall_true_top20_at8_mean": float(gdf["recall_true_top20_at8_raw"].mean()),
        "map_true_top20_at8_mean": float(gdf["map_true_top20_at8"].mean()),
        "ndcg_alpha_at8_mean": float(gdf["ndcg_alpha_at8"].mean()),
        "ndcg_alpha_at20_mean": float(gdf["ndcg_alpha_at20"].mean()),
        "precision_lift_top20_at8_mean": float(gdf["precision_lift_top20_at8"].mean()),
        "boardcap_top8_alpha_mean": float(gdf["boardcap_top8_alpha"].mean()),
        "boardcap_top8_edge_mean": float(gdf["boardcap_top8_edge"].mean()),
        "boardcap_top8_edge_annualized": annualized_return(ret),
        "boardcap_top8_edge_cumulative": float((1.0 + ret).prod() - 1.0) if len(ret) else np.nan,
        "boardcap_top8_edge_mdd": calc_mdd(ret),
        "boardcap_top8_edge_win_rate": float((ret > 0).mean()) if len(ret) else np.nan,
        "boardcap_top8_edge_worst_month": float(ret.min()) if len(ret) else np.nan,
        "top_bottom8_spread_mean": float(gdf["top_bottom8_spread"].mean()),
    }


summary_rows = []
for variant, gdf in monthly_metrics_df.groupby("variant"):
    row = {"variant": variant}
    row.update(summarize_variant(gdf.sort_values(DATE_COL)))
    summary_rows.append(row)
summary_df = pd.DataFrame(summary_rows)

fold_rows = []
for (variant, fold_id), gdf in monthly_metrics_df.groupby(["variant", "fold_id"]):
    row = {"variant": variant, "fold_id": fold_id}
    row.update(summarize_variant(gdf.sort_values(DATE_COL)))
    fold_rows.append(row)
fold_summary_df = pd.DataFrame(fold_rows)

baseline = monthly_metrics_df[monthly_metrics_df["variant"] == "A0_raw_regression"].copy()
baseline_cols = [DATE_COL, "fold_id", "rank_ic", "pr_auc_true_top20", "precision_true_top20_at8_raw", "ndcg_alpha_at8", "boardcap_top8_edge"]
baseline = baseline[baseline_cols].copy()
baseline = baseline.rename(columns=dict((c, c + "_a0") for c in baseline_cols if c not in [DATE_COL, "fold_id"]))

compare_rows = []
for variant, gdf in monthly_metrics_df.groupby("variant"):
    joined = pd.merge(gdf, baseline, on=[DATE_COL, "fold_id"], how="inner")
    row = {"variant": variant, "paired_months": int(len(joined))}
    for metric in ["rank_ic", "pr_auc_true_top20", "precision_true_top20_at8_raw", "ndcg_alpha_at8", "boardcap_top8_edge"]:
        delta = pd.to_numeric(joined[metric], errors="coerce") - pd.to_numeric(joined[metric + "_a0"], errors="coerce")
        row[metric + "_delta_vs_a0"] = float(delta.mean())
        row[metric + "_win_rate_vs_a0"] = float((delta > 0).mean())
        if metric in ["rank_ic", "boardcap_top8_edge"]:
            lo, hi, ppos = moving_block_bootstrap_mean(delta)
            row[metric + "_delta_boot_ci_low"] = lo
            row[metric + "_delta_boot_ci_high"] = hi
            row[metric + "_delta_boot_prob_positive"] = ppos
    compare_rows.append(row)
comparison_df = pd.DataFrame(compare_rows)

summary_df.to_csv(OUT_DIR / "v83_variant_summary.csv", index=False)
fold_summary_df.to_csv(OUT_DIR / "v83_fold_summary.csv", index=False)
comparison_df.to_csv(OUT_DIR / "v83_comparison_vs_a0.csv", index=False)
display_df(summary_df, 20)
display_df(comparison_df, 20)


## 9. 健康指标与实际收益关系


In [ ]:
health_metric_cols = [
    "rank_ic", "roc_auc_true_top20", "pr_auc_true_top20",
    "precision_true_top20_at8_raw", "recall_true_top20_at8_raw",
    "map_true_top20_at8", "ndcg_alpha_at8", "ndcg_alpha_at20",
]
health_rows = []
for variant, gdf in monthly_metrics_df.groupby("variant"):
    g = gdf.sort_values(DATE_COL).copy()
    for metric in health_metric_cols:
        now_corr = safe_rank_ic(g[metric], g["boardcap_top8_edge"])
        next_corr = safe_rank_ic(g[metric].iloc[:-1], g["boardcap_top8_edge"].iloc[1:]) if len(g) > 3 else np.nan
        health_rows.append({
            "variant": variant, "health_metric": metric,
            "same_month_rank_corr_with_top8_edge": now_corr,
            "lag1_rank_corr_with_next_month_top8_edge": next_corr,
            "months": int(len(g)),
        })
health_predictiveness_df = pd.DataFrame(health_rows)
health_predictiveness_df.to_csv(OUT_DIR / "v83_health_metric_predictiveness.csv", index=False)
display_df(health_predictiveness_df, 60)

decision_rows = []
a0 = summary_df[summary_df["variant"] == "A0_raw_regression"]
if len(a0):
    a0 = a0.iloc[0]
    for _, row in summary_df.iterrows():
        variant = row["variant"]
        cmp = comparison_df[comparison_df["variant"] == variant]
        cmp_row = cmp.iloc[0] if len(cmp) else None
        gates = {
            "rank_ic_not_worse": bool(row["rank_ic_mean"] >= a0["rank_ic_mean"]),
            "pr_auc_not_worse": bool(row["pr_auc_true_top20_mean"] >= a0["pr_auc_true_top20_mean"]),
            "precision_not_worse": bool(row["precision_true_top20_at8_mean"] >= a0["precision_true_top20_at8_mean"]),
            "ndcg_not_worse": bool(row["ndcg_alpha_at8_mean"] >= a0["ndcg_alpha_at8_mean"]),
            "top8_edge_not_worse": bool(row["boardcap_top8_edge_mean"] >= a0["boardcap_top8_edge_mean"]),
            "mdd_not_worse": bool(row["boardcap_top8_edge_mdd"] >= a0["boardcap_top8_edge_mdd"]),
        }
        gate_count = 0
        for gate_value in gates.values():
            if gate_value:
                gate_count += 1
        boot_low = np.nan if cmp_row is None else cmp_row.get("boardcap_top8_edge_delta_boot_ci_low", np.nan)
        if variant == "A0_raw_regression":
            decision = "baseline"
        elif gate_count == len(gates) and not pd.isnull(boot_low) and boot_low > 0:
            decision = "promote_candidate"
        elif gate_count >= 4:
            decision = "watch"
        else:
            decision = "reject"
        out = {"variant": variant, "gate_pass_count": gate_count, "gate_total": len(gates), "decision": decision, "top8_edge_delta_boot_ci_low": boot_low}
        out.update(gates)
        decision_rows.append(out)
decision_df = pd.DataFrame(decision_rows)
decision_df.to_csv(OUT_DIR / "v83_pre_registered_decision_table.csv", index=False)
display_df(decision_df, 20)


## 10. 可视化仪表板


In [ ]:
variants = [v for v in RUN_VARIANTS if v in set(summary_df["variant"])]
x = np.arange(len(variants))

# 1) Core metric dashboard.
fig, axes = plt.subplots(2, 4, figsize=(19, 8))
dashboard_metrics = [
    ("rank_ic_mean", "Mean RankIC", 0.0),
    ("roc_auc_true_top20_mean", "ROC-AUC: true Top20", 0.5),
    ("pr_auc_true_top20_mean", "PR-AUC: true Top20", None),
    ("precision_true_top20_at8_mean", "Precision@8: true Top20", None),
    ("recall_true_top20_at8_mean", "Recall@8: true Top20", None),
    ("map_true_top20_at8_mean", "MAP@8: true Top20", None),
    ("ndcg_alpha_at8_mean", "NDCG@8", None),
    ("boardcap_top8_edge_mean", "Monthly Top8 edge", 0.0),
]
for ax, item in zip(axes.ravel(), dashboard_metrics):
    col, title, ref = item
    vals = []
    for v in variants:
        vals.append(float(summary_df.loc[summary_df["variant"] == v, col].iloc[0]))
    ax.bar(x, vals, color=[COLORS.get(v, "#777777") for v in variants])
    if ref is not None:
        ax.axhline(ref, color="#555555", linestyle="--", linewidth=1)
    ax.set_title(title)
    ax.set_xticks(x)
    ax.set_xticklabels([v.split("_")[0] for v in variants])
    ax.grid(axis="y", alpha=0.25)
save_show(fig, "v83_core_metric_dashboard.png")

# 2) Pooled ROC and PR curves; scores are normalized to monthly rank percentiles first.
curve_rows = []
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for v in variants:
    d = curve_panel_df[curve_panel_df["variant"] == v]
    curve = binary_curve_points(d["true_top20"], -pd.to_numeric(d["score_rank_pct"], errors="coerce"))
    if len(curve) == 0:
        continue
    auc = roc_auc_binary(d["true_top20"], -pd.to_numeric(d["score_rank_pct"], errors="coerce"))
    ap = average_precision_binary(d["true_top20"], -pd.to_numeric(d["score_rank_pct"], errors="coerce"))
    axes[0].plot(curve["fpr"], curve["tpr"], color=COLORS.get(v), label="%s AUC=%.3f" % (v.split("_")[0], auc))
    axes[1].plot(curve["recall"], curve["precision"], color=COLORS.get(v), label="%s AP=%.3f" % (v.split("_")[0], ap))
    sampled = curve.iloc[::_bi.max(1, int(len(curve) / 200))].copy()
    sampled["variant"] = v
    sampled["pooled_auc"] = auc
    sampled["pooled_ap"] = ap
    curve_rows.append(sampled)
prevalence = float(curve_panel_df["true_top20"].mean())
axes[0].plot([0, 1], [0, 1], "--", color="#777777", label="random")
axes[0].set_title("Pooled ROC: true future Top20")
axes[0].set_xlabel("False positive rate")
axes[0].set_ylabel("True positive rate")
axes[1].axhline(prevalence, linestyle="--", color="#777777", label="random prevalence")
axes[1].set_title("Pooled Precision-Recall: true future Top20")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
for ax in axes:
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)
save_show(fig, "v83_pooled_roc_pr_curves.png")
curve_points_df = pd.concat(curve_rows, ignore_index=True) if curve_rows else pd.DataFrame()
curve_points_df.to_csv(OUT_DIR / "v83_pooled_curve_points.csv", index=False)

# 3) Cumulative realized Top8 edge and rolling RankIC.
fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)
for v in variants:
    d = monthly_metrics_df[monthly_metrics_df["variant"] == v].sort_values(DATE_COL)
    axes[0].plot(d[DATE_COL], calc_nav(d["boardcap_top8_edge"]).values, color=COLORS.get(v), label=v.split("_")[0])
    axes[1].plot(d[DATE_COL], d["rank_ic"].rolling(6, min_periods=3).mean(), color=COLORS.get(v), label=v.split("_")[0])
axes[0].set_title("Cumulative Top8 edge (proxy)")
axes[0].set_ylabel("NAV")
axes[1].set_title("Rolling 6-month RankIC")
axes[1].axhline(0, color="#555555", linewidth=1)
axes[1].set_ylabel("RankIC")
for ax in axes:
    ax.grid(alpha=0.25)
    ax.legend(ncol=len(variants), fontsize=8)
save_show(fig, "v83_oos_nav_and_rankic.png")

# 4) Distribution stability.
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
rank_data = [monthly_metrics_df.loc[monthly_metrics_df["variant"] == v, "rank_ic"].dropna().values for v in variants]
edge_data = [monthly_metrics_df.loc[monthly_metrics_df["variant"] == v, "boardcap_top8_edge"].dropna().values for v in variants]
axes[0].boxplot(rank_data, labels=[v.split("_")[0] for v in variants], showmeans=True)
axes[1].boxplot(edge_data, labels=[v.split("_")[0] for v in variants], showmeans=True)
axes[0].set_title("Monthly RankIC distribution")
axes[1].set_title("Monthly Top8 edge distribution")
for ax in axes:
    ax.axhline(0, color="#777777", linestyle="--", linewidth=1)
    ax.grid(axis="y", alpha=0.25)
save_show(fig, "v83_metric_distributions.png")

# 5) Score bucket monotonicity. Bucket 1 is highest predicted score.
bucket_summary_df = bucket_monthly_df.groupby(["variant", "score_bucket"])["mean_alpha"].mean().reset_index()
bucket_summary_df.to_csv(OUT_DIR / "v83_bucket_summary.csv", index=False)
fig, ax = plt.subplots(figsize=(11, 5.5))
for v in variants:
    d = bucket_summary_df[bucket_summary_df["variant"] == v].sort_values("score_bucket")
    ax.plot(d["score_bucket"], d["mean_alpha"], marker="o", color=COLORS.get(v), label=v.split("_")[0])
ax.axhline(0, color="#777777", linestyle="--", linewidth=1)
ax.set_xticks(range(1, BUCKET_N + 1))
ax.set_title("Realized alpha by predicted score bucket (1 = highest)")
ax.set_xlabel("Score bucket")
ax.set_ylabel("Mean monthly alpha")
ax.grid(alpha=0.25)
ax.legend(fontsize=8)
save_show(fig, "v83_score_bucket_profile.png")

# 6) Fold heatmap for realized Top8 edge.
fold_ids = [x["fold_id"] for x in FOLD_PLAN if x["fold_id"] in set(fold_summary_df["fold_id"])]
heat = np.full((len(variants), len(fold_ids)), np.nan)
for i, v in enumerate(variants):
    for j, f in enumerate(fold_ids):
        d = fold_summary_df[(fold_summary_df["variant"] == v) & (fold_summary_df["fold_id"] == f)]
        if len(d):
            heat[i, j] = float(d["boardcap_top8_edge_mean"].iloc[0])
fig, ax = plt.subplots(figsize=(12, 4.8))
im = ax.imshow(heat, aspect="auto", cmap="RdYlGn")
ax.set_xticks(np.arange(len(fold_ids)))
ax.set_xticklabels(fold_ids, rotation=30, ha="right")
ax.set_yticks(np.arange(len(variants)))
ax.set_yticklabels([v.split("_")[0] for v in variants])
ax.set_title("Mean monthly Top8 edge by walk-forward fold")
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        if np.isfinite(heat[i, j]):
            ax.text(j, i, "%.3f" % heat[i, j], ha="center", va="center", fontsize=8)
fig.colorbar(im, ax=ax, shrink=0.8)
save_show(fig, "v83_fold_stability_heatmap.png")

# 7) Can ex-post health metrics explain same-month or next-month economics?
same = health_predictiveness_df.pivot(index="health_metric", columns="variant", values="same_month_rank_corr_with_top8_edge")
lag1 = health_predictiveness_df.pivot(index="health_metric", columns="variant", values="lag1_rank_corr_with_next_month_top8_edge")
same = same.reindex(columns=variants)
lag1 = lag1.reindex(index=same.index, columns=variants)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, frame, title in [(axes[0], same, "Same-month metric vs Top8 edge"), (axes[1], lag1, "Metric vs next-month Top8 edge")]:
    arr = frame.values.astype(float)
    im = ax.imshow(arr, aspect="auto", cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_xticks(np.arange(len(frame.columns)))
    ax.set_xticklabels([v.split("_")[0] for v in frame.columns], rotation=30, ha="right")
    ax.set_yticks(np.arange(len(frame.index)))
    ax.set_yticklabels(frame.index, fontsize=8)
    ax.set_title(title)
    for i in range(arr.shape[0]):
        for j in range(arr.shape[1]):
            if np.isfinite(arr[i, j]):
                ax.text(j, i, "%.2f" % arr[i, j], ha="center", va="center", fontsize=7)
fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.75)
save_show(fig, "v83_health_metric_validity.png")


## 11. 结果文件与解读提醒


In [ ]:
readme = [
    "V83 interpretation notes",
    "1. ROC-AUC measures broad discrimination and can look acceptable even when Top8 precision is weak.",
    "2. PR-AUC must be compared with prevalence; it is more informative for rare true-Top20 positives.",
    "3. Precision and recall use the same hit count when K is fixed, so they are not independent evidence.",
    "4. RankIC evaluates the full cross-section; NDCG/MAP emphasize the head; portfolio edge evaluates economic value.",
    "5. Same-month health correlations are ex-post diagnostics. Only lag-1 correlations test limited monitoring value.",
    "6. Top8 returns are offline monthly proxies. Final promotion still requires the aligned JoinQuant backtest.",
    "7. A challenger should not be promoted from one metric, one fold, or a confidence interval crossing zero.",
]
with open(OUT_DIR / "v83_README.txt", "w") as f:
    f.write("\n".join(readme))

print("saved outputs:")
for fp in _bi.sorted(OUT_DIR.glob("v83_*")):
    print("-", fp)
print("figures:")
for fp in _bi.sorted(FIG_DIR.glob("*.png")):
    print("-", fp)
print("\nKey reminder: AUC is not a substitute for Top8 precision or realized Top8 edge.")
